# Qwen MoE Experts Fusion — полный эксперимент

Самодостаточный ноутбук для воспроизведения пайплайна сжатия экспертов модели
`Qwen/Qwen1.5-MoE-A2.7B-Chat` от начала до конца.

**Требования:** GPU с CUDA, ~24+ GB VRAM (для всех 24 слоёв), PyTorch, transformers, datasets, matplotlib, seaborn, tqdm, numpy.

**Этапы:**
1. Матрица совместных активаций экспертов
2. Разбиение экспертов на пары
3. Сохранение весов экспертов и gate
4. Сохранение входов MLP-слоёв
5. Индексы активаций для каждой пары
6. Сжатие экспертов (fusion)
7. Бенчмарк fused vs original слоя
8. Примеры генерации до/после
9. Оценка точности на QA-задачах

Все артефакты сохраняются в `DATA_DIR`. Запускайте ячейки последовательно сверху вниз.


In [ ]:
import gc
import os
import random
import re
import time
import warnings
from collections import defaultdict
from datetime import datetime
from typing import Optional

import datasets
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

import torch

DATA_DIR = "./data_MMLU_1k"

MODEL_NAME = "Qwen/Qwen1.5-MoE-A2.7B-Chat"
EXPERTS = 60
TRAIN_DATASET_NAME = {"path": "cais/mmlu", "name": "all", "split": "auxiliary_train"}
TEST_DATASET_NAME = {
    "path": "allenai/winogrande",
    "name": "winogrande_debiased",
    "split": "test",
}

TRAIN_SIZE = 1_024
TEST_SIZE = TRAIN_SIZE
VALIDATION_SIZE = 1_000
LAYERS = list(range(24))
TOP_K = 4

REPLACE_LAYERS_SEQUANTIAL = False
REPLACE_BEST_LAYERS_SEQUANTIAL = True
REPLACE_LAYERS_SMART_SEQUANTIAL = False
EVOLUTION_STEPS = 50

SEED = 42

# "activation_correlation", "random", "output_similarity"
PAIRING_METHOD = "activation_correlation"

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU"
device = torch.device("cuda")

FILTER_FUSIONS = None
FILTER_METRICS = None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)
print("Device:", device)
print("Model:", MODEL_NAME)
print("Layers:", len(LAYERS), "experts per layer:", EXPERTS)


## Общие утилиты: MLP и ZipIt


In [ ]:
import torch
from torch import nn

default_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def unite_matrix(mat1: torch.Tensor, mat2: torch.Tensor) -> torch.Tensor:
    """Unite matrix A and B in block format:
    A 0
    0 B"""
    add1 = torch.zeros((mat1.shape[0], mat2.shape[1]), device=mat1.device)
    add2 = torch.zeros((mat2.shape[0], mat1.shape[1]), device=mat1.device)
    mat1 = torch.cat([mat1, add1], dim=1)
    mat2 = torch.cat([add2, mat2], dim=1)
    return torch.cat([mat1, mat2], dim=0)


def build_matches(corr_matrix):
    """Get a matrix of correlations and
    build matching with most sum of correlations (greedy)"""
    matches = []
    n = corr_matrix.shape[0]
    used = torch.zeros(n, dtype=torch.bool)

    sim_matrix = corr_matrix.clone()

    sim_matrix.fill_diagonal_(-float("inf"))

    for _ in range(n // 2):
        _, max_idx = torch.max(sim_matrix.view(-1), 0)
        i = max_idx.item() // n
        j = max_idx.item() % n

        matches.append((i, j))
        used[i] = True
        used[j] = True

        sim_matrix[i, :] = -float("inf")
        sim_matrix[:, i] = -float("inf")
        sim_matrix[j, :] = -float("inf")
        sim_matrix[:, j] = -float("inf")

    del sim_matrix, used
    return matches


class Zipper(nn.Linear):
    """Save all inputs and build M and U matrix
    based on greedy matching in correlation matrix"""

    def __init__(self, features):
        super().__init__(features, features)
        self.M = None
        self.U = None
        self.activations = []

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if len(x.shape) == 1:
            x = torch.stack([x])
        self.activations.append(x)
        return x

    def build_MU(self):
        """Count M and U matrixes, using saved while training activations"""
        assert len(self.activations) > 0
        self.dtype = self.activations[0].dtype
        activations = torch.cat(self.activations, dim=0)
        corr_matrix = torch.corrcoef(activations.T)
        n = corr_matrix.shape[0]

        matches = build_matches(corr_matrix)
        self.M = torch.zeros(n // 2, n).to(self.activations[0].device)
        for idx, (i, j) in enumerate(matches):
            self.M[idx, i] = 0.5
            self.M[idx, j] = 0.5
        self.U = 2 * self.M.T.to(dtype=self.activations[0].dtype)

        self.activations = []
        del activations


class MLP(nn.Module):
    """Base interface for all models"""

    def __init__(self):
        super().__init__()

    def forward_unzip(self, x: torch.Tensor, out=None):
        """Count forward and return features of model 1 and model 2 concated"""
        y = self.forward(x.to(dtype=self.gate.weight.dtype))
        if self.U is not None:
            y = y.to(dtype=self.U.dtype)
            if out is not None:
                return torch.nn.functional.linear(
                    y, torch.chunk(self.U, 2, dim=0)[out]
                ).to(dtype=x.dtype)
            return torch.nn.functional.linear(y, self.U).to(dtype=x.dtype)
        if out is not None:
            return y.to(dtype=x.dtype)
        return y.repeat(*([1] * (y.dim() - 1) + [2])).to(dtype=x.dtype)

    def forward(self, x: torch.Tensor):
        raise NotImplementedError()

    def zip_with(self, other: "MLP"):
        """Make an prezipped model with other"""
        raise NotImplementedError()

    def train_with(self, train_data):
        """Run train data and eval M and U matrix"""
        for example in train_data:
            self.forward(example.to(next(self.parameters()).device))
        self.zip1.build_MU()
        self.zip2.build_MU()

    def zip(self):
        """ZipIt operation"""
        raise NotImplementedError()


class SimpleMLP(MLP):
    """MLP with two hidden linear layers"""

    def __init__(
        self,
        in_features: int,
        hid_features: int,
        out_features: int,
        bias: bool = False,
        device=default_device,
    ):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hid_features, bias=bias, device=device)
        self.zip1 = Zipper(hid_features)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hid_features, out_features, bias=bias, device=device)
        self.zip2 = Zipper(out_features)
        self.zipped = False
        self.U = None
        self.to(device)

    def forward(self, x: torch.Tensor):
        if self.zipped:
            x = torch.cat([x, x], dim=-1)
            x = self.fc1(x)
            x = self.zip1(x)
            x = self.act(x)
            x = self.fc2(x)
            x = self.zip2(x)
        else:
            x = self.fc1(x)
            x = self.act(x)
            x = self.fc2(x)
        return x

    def zip_with(self, other: "SimpleMLP"):
        """Make an prezipped model with other"""
        self.fc1.weight = nn.Parameter(unite_matrix(self.fc1.weight, other.fc1.weight))
        self.fc2.weight = nn.Parameter(unite_matrix(self.fc2.weight, other.fc2.weight))
        self.zipped = True

    def zip(self):
        """ZipIt operation"""
        self.U = nn.Parameter(self.zip2.U)

        self.fc1.weight = nn.Parameter(
            self.zip1.M
            @ self.fc1.weight
            @ torch.cat(
                2 * [torch.eye(self.fc1.in_features, device=self.fc1.weight.device)]
            )
        )
        self.fc2.weight = nn.Parameter(self.zip2.M @ self.fc2.weight @ self.zip1.U)

        del self.zip1
        self.zip1 = None
        del self.zip2
        self.zip2 = None

        self.zipped = False


class AdvancedMLP(MLP):
    """MLP with up, down and proj linear layers"""

    def __init__(
        self,
        in_features: int,
        hid_features: int,
        out_features: int,
        bias: bool = False,
        device=default_device,
    ):
        super().__init__()
        self.in_features = in_features
        self.hid_features = hid_features
        self.out_features = out_features
        self.gate = nn.Linear(in_features, hid_features, bias=bias, device=device)
        self.act = nn.SiLU()
        self.up = nn.Linear(in_features, hid_features, bias=bias, device=device)
        self.zip1 = None
        self.down = nn.Linear(hid_features, out_features, bias=bias, device=device)
        self.zip2 = None
        self.zipped = False
        self.U = None
        self.to(device)

    def forward(self, x: torch.Tensor):
        dtype = x.dtype
        x = x.to(dtype=self.gate.weight.dtype)
        if self.zipped:
            x = torch.cat([x, x], dim=-1)
            gate = self.act(self.gate(x))
            up = self.up(x)
            x = torch.mul(gate, up)
            x = self.zip1(x)
            x = self.down(x)
            x = self.zip2(x)
        else:
            x = self.act(self.gate(x)) * self.up(x)
            if self.down is not None:
                x = self.down(x)
        return x.to(dtype=dtype)

    def zip_with(self, other: "MLP"):
        """Make an prezipped model with other"""
        self.zip1 = Zipper(self.hid_features)
        self.zip2 = Zipper(self.out_features)
        self.gate.weight = nn.Parameter(
            unite_matrix(self.gate.weight, other.gate.weight)
        )
        self.up.weight = nn.Parameter(unite_matrix(self.up.weight, other.up.weight))
        self.down.weight = nn.Parameter(
            unite_matrix(self.down.weight, other.down.weight)
        )
        self.zipped = True

    def zip(self):
        """ZipIt operation"""
        self.U = nn.Parameter(self.zip2.U.to(dtype=self.gate.weight.dtype))

        self.gate.weight = nn.Parameter(
            self.zip1.M
            @ self.gate.weight
            @ torch.cat(
                2 * [torch.eye(self.gate.in_features, device=self.gate.weight.device)]
            )
        )
        self.up.weight = nn.Parameter(
            self.zip1.M
            @ self.up.weight
            @ torch.cat(
                2 * [torch.eye(self.up.in_features, device=self.gate.weight.device)]
            )
        )
        self.down.weight = nn.Parameter(self.zip2.M @ self.down.weight @ self.zip1.U)

        del self.zip1
        self.zip1 = None
        del self.zip2
        self.zip2 = None

        self.zipped = False


## Утилиты слияния экспертов и метрик


In [ ]:
import copy
import time
import torch
from tqdm import tqdm
import torch.nn.functional as F
from torch import nn


default_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FUSION_METHODS = {}


def _as_tensor(data):
    if torch.is_tensor(data):
        return data
    return torch.stack(data, dim=0)


def _as_2d_tensor(data, device=None, dtype=None):
    tensor = _as_tensor(data)
    if tensor.dim() == 1:
        tensor = tensor.unsqueeze(0)
    if device is not None or dtype is not None:
        tensor = tensor.to(
            device=device if device is not None else tensor.device, dtype=dtype
        )
    return tensor


def _model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device


def _hidden_activations(model: AdvancedMLP, x: torch.Tensor) -> torch.Tensor:
    x = x.to(device=_model_device(model), dtype=model.gate.weight.dtype)
    return model.act(model.gate(x)) * model.up(x)


def _channel_scores(model: AdvancedMLP, x: torch.Tensor) -> torch.Tensor:
    hidden = _hidden_activations(model, x)
    act_score = hidden.float().pow(2).mean(dim=0).sqrt()
    weight_score = (
        model.gate.weight.float().norm(dim=1)
        + model.up.weight.float().norm(dim=1)
        + model.down.weight.float().norm(dim=0)
    )
    return act_score + 0.05 * weight_score


def _rank_hidden_channels(model: AdvancedMLP, x: torch.Tensor) -> torch.Tensor:
    scores = _channel_scores(model, x)
    return torch.argsort(scores, descending=True)


def _reorder_advanced_mlp(
    model: AdvancedMLP, order: torch.Tensor, create_new: bool = True
) -> AdvancedMLP:
    new_mlp = model if not create_new else copy.deepcopy(model)
    with torch.no_grad():
        new_mlp.gate.weight = nn.Parameter(model.gate.weight[order].clone())
        new_mlp.up.weight = nn.Parameter(model.up.weight[order].clone())
        new_mlp.down.weight = nn.Parameter(model.down.weight[:, order].clone())
    return new_mlp


def count_mse(f, x, y):
    """Count MSE if f(x) and y"""
    return torch.mean((f(x) - y) ** 2)


def eval_merger(
    model1: MLP, model2: MLP, merged_model: MLP, data: torch.Tensor
) -> torch.Tensor:
    """Count MSE of zipped model with two simple models"""
    with torch.no_grad():
        return count_mse(
            merged_model.forward_unzip,
            data,
            torch.cat([model1.forward(data), model2.forward(data)], dim=-1),
        )


def zipMLP(
    mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True
) -> MLP:
    """Take two MLP models, zipit using train data. Return zipped MLP"""
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)
    with torch.no_grad():
        new_mlp = mlpA
        if create_new:
            new_mlp = copy.deepcopy(mlpA)
        new_mlp.zip_with(mlpB)
        new_mlp.train_with(train_data)
        new_mlp.zip()
    return new_mlp


def avgMLP(
    mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True
) -> MLP:
    """Take two MLP models. Return same model with parameters averaged"""
    with torch.no_grad():
        new_mlp = mlpA
        if create_new:
            new_mlp = copy.deepcopy(mlpA)
        new_dict = dict()
        dictA = mlpA.state_dict()
        dictB = mlpB.state_dict()
        for key in dictA.keys():
            new_dict[key] = (dictA[key] + dictB[key]) / 2
        new_mlp.load_state_dict(new_dict)
    return new_mlp


def procrustesMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
) -> AdvancedMLP:
    """Align hidden channels by rank of activation signatures, then average.

    This is a lightweight, architecture-safe substitute for a full orthogonal
    Procrustes alignment: with gated MLP experts we can only permute hidden
    channels without breaking the nonlinearity structure, so we align by
    importance rank instead of applying a dense rotation.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    with torch.no_grad():
        data = _as_2d_tensor(train_data, device=_model_device(mlpA))
        order_a = _rank_hidden_channels(mlpA, data)
        order_b = _rank_hidden_channels(mlpB, data)

        aligned_a = _reorder_advanced_mlp(mlpA, order_a, create_new=True)
        aligned_b = _reorder_advanced_mlp(mlpB, order_b, create_new=True)
        return avgMLP(aligned_a, aligned_b, train_data, create_new=create_new)


def _batch_iter(x: torch.Tensor, batch_size: int):
    for start in range(0, len(x), batch_size):
        yield x[start : start + batch_size]


def _estimate_fisher_diag(
    model: AdvancedMLP,
    data: torch.Tensor,
    teacher_output: torch.Tensor,
    teacher_hidden: torch.Tensor,
    *,
    batch_size: int = 256,
    hidden_loss_weight: float = 0.25,
) -> dict[str, torch.Tensor]:
    params = {name: p for name, p in model.named_parameters() if p.requires_grad}
    fisher = {
        name: torch.zeros_like(p, dtype=torch.float32) for name, p in params.items()
    }

    model.zero_grad(set_to_none=True)
    was_training = model.training
    model.eval()

    with torch.enable_grad():
        for batch_x, batch_teacher_out, batch_teacher_hidden in zip(
            _batch_iter(data, batch_size),
            _batch_iter(teacher_output, batch_size),
            _batch_iter(teacher_hidden, batch_size),
        ):
            model.zero_grad(set_to_none=True)
            pred_out = model(batch_x)
            pred_hidden = _hidden_activations(model, batch_x)
            loss = F.mse_loss(
                pred_out, batch_teacher_out
            ) + hidden_loss_weight * F.mse_loss(pred_hidden, batch_teacher_hidden)
            grads = torch.autograd.grad(
                loss,
                tuple(params.values()),
                retain_graph=False,
                create_graph=False,
                allow_unused=True,
            )
            for (name, _), grad in zip(params.items(), grads):
                if grad is None:
                    continue
                fisher[name] += grad.detach().float().pow(2)

    if was_training:
        model.train()
    return fisher


def fisherMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
) -> AdvancedMLP:
    """Merge experts with parameter-wise Fisher precision weighting.

    We approximate Fisher information by squared gradients of a self-supervised
    distillation objective against the pairwise teacher ensemble.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    with torch.no_grad():
        data = _as_2d_tensor(train_data, device=_model_device(mlpA))
        teacher_output = 0.5 * (mlpA(data) + mlpB(data))
        teacher_hidden = 0.5 * (
            _hidden_activations(mlpA, data) + _hidden_activations(mlpB, data)
        )

    fisher_a = _estimate_fisher_diag(mlpA, data, teacher_output, teacher_hidden)
    fisher_b = _estimate_fisher_diag(mlpB, data, teacher_output, teacher_hidden)

    merged = mlpA if not create_new else copy.deepcopy(mlpA)
    merged_state = merged.state_dict()
    state_a = mlpA.state_dict()
    state_b = mlpB.state_dict()

    for key in merged_state.keys():
        imp_a = fisher_a.get(key)
        imp_b = fisher_b.get(key)
        if imp_a is None or imp_b is None:
            merged_state[key] = (state_a[key] + state_b[key]) / 2
            continue

        imp_a = imp_a.float()
        imp_b = imp_b.float()
        # Normalize each tensor locally to keep the weighting numerically stable.
        imp_a = imp_a / (imp_a.mean() + 1e-8)
        imp_b = imp_b / (imp_b.mean() + 1e-8)
        merged_state[key] = (
            ((imp_a * state_a[key].float()) + (imp_b * state_b[key].float()))
            / (imp_a + imp_b + 1e-8)
        ).to(dtype=state_a[key].dtype)

    merged.load_state_dict(merged_state, strict=True)
    return merged


def tiesMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
    trim_ratio: float = 0.2,
) -> AdvancedMLP:
    """Sparse sign-consensus merge inspired by TIES.

    The original TIES formulation assumes task vectors from a shared base model.
    In this project we only have a pair of experts, so we adapt the idea to a
    pairwise merge: keep strong updates, resolve sign conflicts by magnitude,
    and trim small-magnitude entries.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    with torch.no_grad():
        merged = mlpA if not create_new else copy.deepcopy(mlpA)
        state_a = mlpA.state_dict()
        state_b = mlpB.state_dict()
        merged_state = {}

        for key in state_a.keys():
            wa = state_a[key].float()
            wb = state_b[key].float()
            same_sign = torch.sign(wa) == torch.sign(wb)
            chosen = torch.where(
                same_sign, 0.5 * (wa + wb), torch.where(wa.abs() >= wb.abs(), wa, wb)
            )

            if trim_ratio > 0:
                threshold = torch.quantile(chosen.abs().flatten(), trim_ratio)
                chosen = torch.where(
                    chosen.abs() >= threshold, chosen, torch.zeros_like(chosen)
                )

            merged_state[key] = chosen.to(dtype=state_a[key].dtype)

        merged.load_state_dict(merged_state, strict=True)
        return merged


def distillMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
) -> AdvancedMLP:
    """Train a student expert to mimic the pairwise teacher ensemble.

    This is the most expensive merge rule, but it is usually the strongest when
    the two experts are meaningfully different and you care about actual output
    preservation more than a closed-form weight merge.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    device = _model_device(mlpA)
    data = _as_2d_tensor(train_data, device=device)
    batch_size = min(256, max(1, len(data)))
    student = avgMLP(mlpA, mlpB, train_data, create_new=create_new)

    with torch.no_grad():
        teacher_out = 0.5 * (mlpA(data) + mlpB(data))
        teacher_hidden = 0.5 * (
            _hidden_activations(mlpA, data) + _hidden_activations(mlpB, data)
        )

    student.train()
    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=0.0)
    epochs = 4 if len(data) >= 512 else 8

    with torch.enable_grad():
        for _ in range(epochs):
            for batch_x, batch_teacher_out, batch_teacher_hidden in zip(
                _batch_iter(data, batch_size),
                _batch_iter(teacher_out, batch_size),
                _batch_iter(teacher_hidden, batch_size),
            ):
                optimizer.zero_grad(set_to_none=True)
                pred_out = student(batch_x)
                pred_hidden = _hidden_activations(student, batch_x)
                loss = F.mse_loss(pred_out, batch_teacher_out) + 0.25 * F.mse_loss(
                    pred_hidden, batch_teacher_hidden
                )
                loss.backward()
                optimizer.step()

    student.eval()
    return student


def proj(
    mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True
) -> MLP:
    return copy.deepcopy(mlpA)


def solve_linear(X, Y, lambda_ridge=1e-3):
    d_in = X.shape[1]
    XtX = X.T @ X + lambda_ridge * torch.eye(d_in, device=X.device, dtype=X.dtype)
    rhs = X.T @ Y

    W = torch.linalg.solve(XtX, rhs).T
    return W


def finetune(method):
    def inner(mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True):
        newMlp = method(mlpA, mlpB, train_data, create_new)
        if len(train_data) < 2 * mlpA.down.out_features:
            return newMlp
        newMlp.down = None
        all_tokens = _as_tensor(train_data)
        real_res = torch.cat(
            [mlpA.forward(all_tokens), mlpB.forward(all_tokens)], dim=-1
        )
        new_res = newMlp.forward(all_tokens)
        newMlp.U = torch.nn.Parameter(
            solve_linear(new_res.float(), real_res.float()).to(
                dtype=mlpA.down.weight.dtype
            )
        )
        return newMlp

    return inner


FUSION_METHODS["ZipIt"] = zipMLP
FUSION_METHODS["Average"] = avgMLP
FUSION_METHODS["Projection"] = proj
FUSION_METHODS["Procrustes"] = procrustesMLP
FUSION_METHODS["Fisher"] = fisherMLP
FUSION_METHODS["TIES"] = tiesMLP
FUSION_METHODS["Distillation"] = distillMLP

FUSION_METHODS["FineTune_ZipIt"] = finetune(zipMLP)
FUSION_METHODS["FineTune_Average"] = finetune(avgMLP)
FUSION_METHODS["FineTune_Projection"] = finetune(proj)
FUSION_METHODS["FineTune_Procrustes"] = finetune(procrustesMLP)
FUSION_METHODS["FineTune_Fisher"] = finetune(fisherMLP)
FUSION_METHODS["FineTune_TIES"] = finetune(tiesMLP)
FUSION_METHODS["FineTune_Distillation"] = finetune(distillMLP)


@torch.no_grad()
def find_mlp_mse_err(
    merger, n: int, hidden: int, out: int, examples: int = 10_000, device=default_device
) -> torch.Tensor:
    """For a given merger make two random models,
    merge them, train and test on random data.
    Return it's MSE with original models"""
    with torch.no_grad():
        mlp1 = AdvancedMLP(n, hidden, out, device=device)
        mlp2 = AdvancedMLP(n, hidden, out, device=device)

        x_train = torch.rand((10 * examples, n), device=device)
        x_test = torch.rand((10 * examples, n), device=device)

        mlp_merged = merger(mlp1, mlp2, x_train)

        res = eval_merger(mlp1, mlp2, mlp_merged, x_test)

    del mlp_merged, x_train, x_test, mlp1, mlp2
    torch.cuda.empty_cache()
    return res


def find_mlp_mse_err_repeat(
    merger, n: int, hidden: int, out: int, repeat: int = 1000
) -> torch.Tensor:
    """Repeat previous test [repeat] times. Return a tensor of results"""
    return (
        torch.stack(
            [
                find_mlp_mse_err(merger, n, hidden, out, examples=1000)
                for _ in tqdm(range(repeat))
            ],
            dim=0,
        )
        .detach()
        .cpu()
    )


@torch.no_grad()
def turn_in_mlp(mlp: torch.nn.Module, device=default_device) -> MLP:
    """Extract matrixs from loaded mlp layer
    and create AdvancedMLP with same weights"""
    new_mlp = AdvancedMLP(
        mlp.gate_proj.in_features,
        mlp.gate_proj.out_features,
        mlp.down_proj.out_features,
        device=device,
    )
    new_mlp.gate = mlp.gate_proj
    new_mlp.up = mlp.up_proj
    new_mlp.down = mlp.down_proj
    return new_mlp


def mse(x, y):
    return torch.square(x - y).mean().item()


def cosine(x, y):
    return torch.cosine_similarity(x, y).mean().item()


def mae(x, y):
    return torch.abs(x - y).mean().item()


def rmse(x, y):
    return torch.sqrt(torch.square(x - y).mean()).item()


def relative_mse(x, y):
    denom = torch.square(x - x.mean(dim=0, keepdim=True)).mean().clamp_min(1e-8)
    return (torch.square(x - y).mean() / denom).item()


def r2_score(x, y):
    ss_res = torch.square(x - y).sum()
    ss_tot = torch.square(x - x.mean(dim=0, keepdim=True)).sum().clamp_min(1e-8)
    return (1.0 - ss_res / ss_tot).item()


def pearson_corr(x, y):
    x = x.flatten().float()
    y = y.flatten().float()
    x = x - x.mean()
    y = y - y.mean()
    denom = torch.sqrt((x.square().sum() * y.square().sum()).clamp_min(1e-8))
    return (x.mul(y).sum() / denom).item()


def softmax_kl_divergence(logits_p, logits_q):
    p = F.log_softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)
    return F.kl_div(p, q, reduction="batchmean").item()


def topk_agreement(logits_p, logits_q, k=1):
    k = max(1, min(k, logits_p.shape[-1], logits_q.shape[-1]))
    top_p = torch.topk(logits_p, k=k, dim=-1).indices
    top_q = torch.topk(logits_q, k=k, dim=-1).indices
    if k == 1:
        return (top_p.squeeze(-1) == top_q.squeeze(-1)).float().mean().item()
    overlap = []
    for a, b in zip(top_p, top_q):
        overlap.append(len(set(a.tolist()) & set(b.tolist())) / float(k))
    return float(sum(overlap) / max(1, len(overlap)))


def prediction_entropy(logits):
    probs = F.softmax(logits, dim=-1)
    return (-torch.sum(probs * torch.log(probs.clamp_min(1e-8)), dim=-1)).mean().item()


def center_gram(K):
    n = K.size(0)
    one = torch.ones((n, n), device=K.device) / n
    return K - one @ K - K @ one + one @ K @ one


def linear_cka(X, Y):
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    K = X @ X.T
    L = Y @ Y.T

    Kc = center_gram(K)
    Lc = center_gram(L)

    hsic = (Kc * Lc).sum()

    norm_x = torch.sqrt((Kc * Kc).sum())
    norm_y = torch.sqrt((Lc * Lc).sum())

    return (hsic / (norm_x * norm_y + 1e-8)).mean().item()


def js_divergence_from_logits(logits_p, logits_q):
    """
    logits_p, logits_q: (N, M)
    """
    p = F.softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)

    m = 0.5 * (p + q)

    js = 0.5 * (
        F.kl_div(m.log(), p, reduction="batchmean")
        + F.kl_div(m.log(), q, reduction="batchmean")
    )

    return js.mean().item()


@torch.no_grad()
def timer(mlp: MLP, test_data, iters=3) -> float:
    """Apply mlp on data and return required time"""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    for _ in range(iters):
        mlp.forward_unzip(test_data)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()
    return end_time - start_time


@torch.no_grad()
def test_model_on_data(fused_mlp, mlp1, mlp2, test_data):
    all_tokens = _as_tensor(test_data)
    real_res = torch.cat(
        [mlp1.forward(all_tokens), mlp2.forward(all_tokens)], dim=-1
    ).float()
    new_res = fused_mlp.forward_unzip(all_tokens).float()
    return {
        "MSE": mse(real_res, new_res),
        "RMSE": rmse(real_res, new_res),
        "MAE": mae(real_res, new_res),
        "RelMSE": relative_mse(real_res, new_res),
        "Cosine": cosine(real_res, new_res),
        "Pearson": pearson_corr(real_res, new_res),
        "R2": r2_score(real_res, new_res),
        "CKA": linear_cka(real_res, new_res),
        "JS": js_divergence_from_logits(real_res, new_res),
        "KL": softmax_kl_divergence(real_res, new_res),
        "Top1": topk_agreement(real_res, new_res, k=1),
        "Top5": topk_agreement(real_res, new_res, k=min(5, real_res.shape[-1])),
        "EntropyGap": abs(prediction_entropy(real_res) - prediction_entropy(new_res)),
        "Time": 1000 * timer(fused_mlp, all_tokens),
    }


# True is greater is better
best_metrics = {
    "MSE": False,
    "RMSE": False,
    "MAE": False,
    "RelMSE": False,
    "Cosine": True,
    "Pearson": True,
    "R2": True,
    "JS": False,
    "KL": False,
    "Top1": True,
    "Top5": True,
    "EntropyGap": False,
    "CKA": True,
}


## Подбор пар экспертов


In [ ]:
import os
from typing import Optional
import torch
from tqdm import tqdm


def random_pairs(n_experts: int) -> list[tuple[int, int]]:
    perm = torch.randperm(n_experts).tolist()
    return [(perm[2 * i], perm[2 * i + 1]) for i in range(n_experts // 2)]


def eval_match(sim_matrix: torch.Tensor, matches, top_k: int = 8) -> float:
    win = 0.0
    for i, j in matches:
        win += float(sim_matrix[i, j])
    return win / (float(sim_matrix.sum()) / top_k)


def evolutionary_match(
    sim_matrix: torch.Tensor,
    evolution_steps: int,
    progress: bool = False,
) -> list[tuple[int, int]]:
    assert sim_matrix.dim() == 2 and sim_matrix.shape[0] == sim_matrix.shape[1]
    n = sim_matrix.shape[0]

    best_win = -float("inf")
    best_pairs: list[tuple[int, int]] = []

    iterator = range(evolution_steps)
    if progress:
        iterator = tqdm(iterator)

    with torch.no_grad():
        for _ in iterator:
            perm = torch.randperm(n).tolist()
            pairs = set((perm[2 * i], perm[2 * i + 1]) for i in range(n // 2))
            friend = [-1] * n
            for i, j in pairs:
                friend[i] = j
                friend[j] = i

            def swap(i, j):
                if (i, friend[i]) in pairs:
                    pairs.remove((i, friend[i]))
                else:
                    pairs.remove((friend[i], i))
                if (j, friend[j]) in pairs:
                    pairs.remove((j, friend[j]))
                else:
                    pairs.remove((friend[j], j))
                pairs.add((i, j))
                pairs.add((friend[i], friend[j]))
                fi, fj = friend[i], friend[j]
                friend[fi], friend[fj] = fj, fi
                friend[i], friend[j] = j, i

            def win_from_swap(i, j):
                return (
                    sim_matrix[i, j]
                    + sim_matrix[friend[i], friend[j]]
                    - sim_matrix[i, friend[i]]
                    - sim_matrix[j, friend[j]]
                ).item()

            while True:
                delta = 0.0
                bi, bj = -1, -1
                for i in range(n):
                    for j in range(i + 1, n):
                        d = win_from_swap(i, j)
                        if d > delta:
                            delta = d
                            bi, bj = i, j
                if delta < 1e-6:
                    break
                swap(bi, bj)

            win = eval_match(sim_matrix, pairs)
            if win > best_win:
                best_win = win
                best_pairs = list(pairs)

    return best_pairs


def compute_activation_correlation_matrix(layer: int, data_dir: str) -> torch.Tensor:
    path = f"{data_dir}/activations/train_matrix_{layer}.pt"
    matrix = torch.load(path).cpu().float()
    matrix = matrix / matrix.sum()
    return matrix


def compute_output_similarity_matrix(
    layer: int,
    data_dir: str,
    n_experts: int,
    device: torch.device,
    num_samples: int = 4096,
    batch_size: int = 512,
) -> torch.Tensor:
    experts_dir = f"{data_dir}/experts/{layer}"
    inputs_path = f"{data_dir}/mlp_inputs/train_{layer}.pt"
    if not os.path.isdir(experts_dir):
        raise FileNotFoundError("PAIRING_METHOD='output_similarity' requires step 3 ")
    if not os.path.isfile(inputs_path):
        raise FileNotFoundError("PAIRING_METHOD='output_similarity' requires step 4 ")

    raw_inputs = torch.load(inputs_path)
    if isinstance(raw_inputs, list):
        inputs = torch.stack(raw_inputs, dim=0)
    else:
        inputs = raw_inputs
    if inputs.dim() == 3:
        inputs = inputs.reshape(-1, inputs.shape[-1])
    if inputs.shape[0] > num_samples:
        idx = torch.randperm(inputs.shape[0])[:num_samples]
        inputs = inputs[idx]
    inputs = inputs.to(device)

    norm_outputs: list[torch.Tensor] = []
    for i in range(n_experts):
        expert = torch.load(f"{experts_dir}/expert_{i}.pt", map_location=device)
        expert.eval()
        outs = []
        with torch.no_grad():
            for start in range(0, inputs.shape[0], batch_size):
                chunk = inputs[start : start + batch_size]
                out = expert(chunk).float()
                out = out / out.norm(dim=-1, keepdim=True).clamp_min(1e-8)
                outs.append(out)
        norm_outputs.append(torch.cat(outs, dim=0).cpu())
        del expert
        if device.type == "cuda":
            torch.cuda.empty_cache()

    stacked = torch.stack(norm_outputs, dim=0)  # [E, N, H]
    n = stacked.shape[1]
    sim = torch.einsum("inh,jnh->ij", stacked, stacked) / n
    return sim


def get_pairs(
    method: str,
    layer: int,
    n_experts: int,
    data_dir: str,
    device: Optional[torch.device] = None,
    evolution_steps: int = 0,
    progress: bool = False,
) -> list[tuple[int, int]]:
    if method == "random":
        return random_pairs(n_experts)

    if method == "activation_correlation":
        sim = compute_activation_correlation_matrix(layer, data_dir)
    elif method == "output_similarity":
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sim = compute_output_similarity_matrix(layer, data_dir, n_experts, device)
    else:
        raise ValueError(
            f"Unknown PAIRING_METHOD: {method!r}. "
            f"Expected: 'activation_correlation', 'random', 'output_similarity'."
        )

    if evolution_steps > 0:
        return evolutionary_match(sim, evolution_steps, progress=progress)
    return build_matches(sim)


## FusionLayer и QA-оценка


In [ ]:
from torch import nn
import torch
import torch.nn.functional as F


class FusionLayer(nn.Module):
    def __init__(
        self,
        *,
        experts,
        expert_pairs,
        fusion_experts,
        gate,
        shared_expert,
        shared_expert_gate,
        use_default_experts,
    ):
        super().__init__()
        self.gate = gate
        if use_default_experts:
            self.experts = nn.ModuleList(experts)
        self.expert_pairs = expert_pairs
        self.fusion_experts = nn.ModuleList(fusion_experts)
        self.shared_expert = shared_expert
        self.shared_expert_gate = shared_expert_gate
        self.use_default_experts = use_default_experts

        self.num_experts = len(experts)
        self.pair_idx = [None] * len(experts)
        self.friend = [None] * len(experts)
        for i, (x, y) in enumerate(expert_pairs):
            self.friend[x] = y
            self.friend[y] = x
            self.pair_idx[x] = i
            self.pair_idx[y] = i

    def forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size, sequence_length, hidden_dim = hidden_states.shape
        hidden_states_reshaped = hidden_states.view(-1, hidden_dim)
        shared_expert_output = self.shared_expert(hidden_states_reshaped)
        _, routing_weights, selected_experts = self.gate(hidden_states_reshaped)
        expert_output = self._forward(
            hidden_states_reshaped, selected_experts, routing_weights
        )

        shared_expert_output = (
            F.sigmoid(self.shared_expert_gate(hidden_states_reshaped))
            * shared_expert_output
        )

        expert_output = expert_output + shared_expert_output
        expert_output = expert_output.reshape(batch_size, sequence_length, hidden_dim)
        return expert_output

    def _forward(
        self,
        hidden_states: torch.Tensor,
        top_k_index: torch.Tensor,
        top_k_weights: torch.Tensor,
    ) -> torch.Tensor:
        final_hidden_states = torch.zeros_like(hidden_states)
        with torch.no_grad():
            expert_mask = torch.nn.functional.one_hot(
                top_k_index, num_classes=self.num_experts
            )
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hit:
            expert_idx = expert_idx[0]
            if expert_idx == self.num_experts:
                continue
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current_state = hidden_states[token_idx]
            pair_idx = self.pair_idx[expert_idx]
            pair = self.expert_pairs[pair_idx]
            current_hidden_states = self.fusion_experts[pair_idx].forward_unzip(
                current_state, out=(0 if expert_idx == pair[0] else 1)
            )
            current_hidden_states = (
                current_hidden_states * top_k_weights[token_idx, top_k_pos, None]
            )
            final_hidden_states.index_add_(
                0, token_idx, current_hidden_states.to(final_hidden_states.dtype)
            )

        return final_hidden_states

    def forward_default(
        self, hidden_states: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size, sequence_length, hidden_dim = hidden_states.shape
        hidden_states_reshaped = hidden_states.view(-1, hidden_dim)
        shared_expert_output = self.shared_expert(hidden_states_reshaped)
        _, routing_weights, selected_experts = self.gate(hidden_states_reshaped)
        expert_output = self._forward_default(
            hidden_states_reshaped, selected_experts, routing_weights
        )

        shared_expert_output = (
            F.sigmoid(self.shared_expert_gate(hidden_states_reshaped))
            * shared_expert_output
        )

        expert_output = expert_output + shared_expert_output
        expert_output = expert_output.reshape(batch_size, sequence_length, hidden_dim)
        return expert_output

    def _forward_default(
        self,
        hidden_states: torch.Tensor,
        top_k_index: torch.Tensor,
        top_k_weights: torch.Tensor,
    ) -> torch.Tensor:
        final_hidden_states = torch.zeros_like(hidden_states)
        with torch.no_grad():
            expert_mask = torch.nn.functional.one_hot(
                top_k_index, num_classes=self.num_experts
            )
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hit:
            expert_idx = expert_idx[0]
            if expert_idx == self.num_experts:
                continue
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current_state = hidden_states[token_idx]
            current_hidden_states = self.experts[expert_idx](current_state)
            current_hidden_states = (
                current_hidden_states * top_k_weights[token_idx, top_k_pos, None]
            )
            final_hidden_states.index_add_(
                0, token_idx, current_hidden_states.to(final_hidden_states.dtype)
            )

        return final_hidden_states


def get_fusion_layer_by_index(
    index, experts_dir, device=torch.device("cpu"), use_default_experts=True
):
    expert_pairs = torch.load(f"{DATA_DIR}/pairs/{index}.pt")
    if use_default_experts:
        experts = [
            torch.load(
                f"{DATA_DIR}/experts/{index}/expert_{i}.pt", weights_only=False
            ).to(device)
            for i in range(EXPERTS)
        ]
    else:
        experts = [None] * EXPERTS
    fusion_experts = [
        torch.load(f"{DATA_DIR}/{experts_dir}/{index}/{i}.pt", weights_only=False).to(
            device
        )
        for i in range(len(expert_pairs))
    ]
    gate = torch.load(f"{DATA_DIR}/experts/{index}/gate.pt", weights_only=False).to(
        device
    )
    shared_expert = torch.load(
        f"{DATA_DIR}/experts/{index}/shared_expert.pt", weights_only=False
    ).to(device)
    shared_expert_gate = torch.load(
        f"{DATA_DIR}/experts/{index}/shared_expert_gate.pt", weights_only=False
    ).to(device)

    return FusionLayer(
        experts=experts,
        expert_pairs=expert_pairs,
        fusion_experts=fusion_experts,
        gate=gate,
        shared_expert=shared_expert,
        shared_expert_gate=shared_expert_gate,
        use_default_experts=use_default_experts,
    )


import random

import torch
from datasets import load_dataset


_DATA_CACHE: dict = {}


def _cache_key(dataset_kwargs: dict, n: int) -> tuple:
    return (tuple(sorted(dataset_kwargs.items())), n)


def _join_or(items: list) -> str:
    if len(items) <= 1:
        return "".join(items)
    if len(items) == 2:
        return f"{items[0]} or {items[1]}"
    return ", ".join(items[:-1]) + f", or {items[-1]}"


def _extract_question(sample: dict) -> str:
    if "question" in sample:
        return sample["question"]
    if "sentence" in sample:  # winogrande
        return sample["sentence"]
    if "question_stem" in sample:  # openbookqa
        return sample["question_stem"]
    raise KeyError("sample has neither 'question' nor 'sentence'")


def _extract_choices(sample: dict) -> tuple:
    if "choices" in sample:
        choices = sample["choices"]
        if isinstance(choices, dict):  # commonsense_qa, ARC, OpenBookQA
            return list(choices["label"]), list(choices["text"])
        labels = [chr(ord("A") + i) for i in range(len(choices))]  # mmlu
        return labels, list(choices)
    if "option1" in sample and "option2" in sample:  # winogrande
        return ["A", "B"], [sample["option1"], sample["option2"]]
    raise KeyError("sample has no recognized choices field")


def _extract_answer(sample: dict, labels: list) -> str:
    if "answerKey" in sample:
        return sample["answerKey"]
    ans = sample["answer"]
    if isinstance(ans, int):  # mmlu
        return labels[ans]
    if ans in labels:  # already a letter
        return ans
    return labels[int(ans) - 1]  # winogrande: '1' -> 'A', '2' -> 'B'


def _format_prompt(sample: dict, tokenizer, letters: list) -> str:
    labels, texts = _extract_choices(sample)
    options_text = ""
    for label, text in zip(labels, texts):
        options_text += f"{label}. {text}\n"

    system_message = (
        "You are a precise evaluation system. "
        "You will be given a question and multiple-choice options. "
        f"Your task is to answer with ONLY the letter corresponding to the correct option ({_join_or(letters)}). "
        "Do not write the full answer, do not write explanations. Output only one character."
    )
    user_message = (
        f"Question: {_extract_question(sample)}\nOptions:\n{options_text}\nAnswer:"
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _resolve_answer_token_ids(tokenizer, letters: list) -> list:
    ids = []
    for letter in letters:
        toks = tokenizer.encode(letter, add_special_tokens=False)
        ids.append(toks[0])
    return ids


def _prepare(dataset_kwargs: dict, n: int, tokenizer, seed: int) -> dict:
    full = load_dataset(**dataset_kwargs)
    n = min(n, len(full))
    indices = sorted(random.Random(seed).sample(range(len(full)), n))
    subset = full.select(indices)

    # Per-dataset letter set: assume every sample shares the schema of subset[0].
    answer_letters, _ = _extract_choices(subset[0])
    letter_to_idx = {letter: i for i, letter in enumerate(answer_letters)}

    prompts = [_format_prompt(s, tokenizer, answer_letters) for s in subset]
    ground_truths = [_extract_answer(s, answer_letters) for s in subset]
    answer_token_ids = _resolve_answer_token_ids(tokenizer, answer_letters)

    return {
        "prompts": prompts,
        "ground_truths": ground_truths,
        "answer_letters": answer_letters,
        "letter_to_idx": letter_to_idx,
        "answer_token_ids": answer_token_ids,
    }


@torch.no_grad()
def evaluate_qa(
    model,
    tokenizer,
    dataset_kwargs: dict,
    n: int,
    *,
    seed: int = 42,
    batch_size: int = 16,
    max_length: int = 512,
) -> float:
    set_seed(seed)
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    was_training = model.training
    model.eval()
    try:
        key = _cache_key(dataset_kwargs, n)
        cache = _DATA_CACHE.get(key)
        if cache is None:
            cache = _prepare(dataset_kwargs, n, tokenizer, seed)
            _DATA_CACHE[key] = cache

        prompts = cache["prompts"]
        ground_truths = cache["ground_truths"]
        letter_to_idx = cache["letter_to_idx"]
        answer_token_tensor = torch.tensor(
            cache["answer_token_ids"], device=model.device
        )

        correct = 0
        total = 0
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i : i + batch_size]
            batch_truths = ground_truths[i : i + batch_size]

            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(model.device)

            outputs = model(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                output_hidden_states=False,
                output_attentions=False,
                use_cache=False,
            )
            last_logits = outputs.logits[:, -1, :]

            idx = answer_token_tensor.unsqueeze(0).expand(last_logits.size(0), -1)
            answer_logits = torch.gather(last_logits, 1, idx)
            predicted = torch.argmax(answer_logits, dim=1)
            truth = torch.tensor(
                [letter_to_idx[t] for t in batch_truths], device=model.device
            )

            correct += (predicted == truth).sum().item()
            total += len(batch_prompts)

            del inputs, outputs, last_logits, idx, answer_logits, predicted, truth
            torch.cuda.empty_cache()

        return correct / total
    finally:
        if was_training:
            model.train()


## Этап 1. Матрица совместных активаций


In [ ]:
import torch
import torch.nn as nn
import os
from tqdm import tqdm
import transformers
import datasets

print("Device:", device)

set_seed(SEED)

if "model" not in globals():
    model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map=device
    )
if "tokenizer" not in globals():
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

# Remove extra layers
MAX_LAYERS = max(LAYERS) + 1
for i in range(MAX_LAYERS, len(model.model.layers)):
    del model.model.layers[MAX_LAYERS]

torch.cuda.empty_cache()
model = model.to(device)

if "_train_data" not in globals():
    _train_data = datasets.load_dataset(**TRAIN_DATASET_NAME)["question"]
if "_test_data" not in globals():
    _test_data = datasets.load_dataset(**TEST_DATASET_NAME)["question"]
train_data = _train_data[:TRAIN_SIZE]
test_data = _test_data[:TEST_SIZE]
print("Example of train data:", train_data[0])
print("Example of test data:", test_data[0])

class ExpertsLogger(nn.Module):
    """Simple class for saving activated experts based on gate"""

    def __init__(self, gate: nn.Module):
        super().__init__()
        self.gate = gate
        self.num_experts = gate.num_experts
        self.experts: torch.Tensor | None = torch.zeros(
            (self.num_experts, self.num_experts)
        )

    def forward(self, x: torch.Tensor):
        router_logits = self.gate(x)

        if isinstance(router_logits, tuple):
            routing_weights = router_logits[0]
            selected_experts = router_logits[-1]
        else:
            routing_weights = router_logits.softmax(dim=1, dtype=torch.float)
            _, selected_experts = torch.topk(routing_weights, TOP_K, dim=-1)
        if self.experts is not None:
            for experts_stack in selected_experts:
                for expert1 in experts_stack:
                    for expert2 in experts_stack:
                        self.experts[expert1, expert2] += 1

        del routing_weights, selected_experts
        torch.cuda.empty_cache()

        return router_logits

    def clear_experts(self):
        self.experts = torch.zeros((self.num_experts, self.num_experts))

    def get_experts(self):
        return self.experts

    def freeze(self):
        self.experts = None

loggers = dict()
for i in LAYERS:
    if isinstance(model.model.layers[i].mlp.gate, ExpertsLogger):
        loggers[i] = ExpertsLogger(model.model.layers[i].mlp.gate.gate)
    else:
        loggers[i] = ExpertsLogger(model.model.layers[i].mlp.gate)
    model.model.layers[i].mlp.gate = loggers[i]

def get_experts(data, loggers):
    with torch.no_grad():
        for logger in loggers.values():
            logger.clear_experts()
        for example in tqdm(data):
            tokenized = tokenizer(example, return_tensors="pt")
            input_ids = tokenized["input_ids"].to(device)
            res = model(input_ids)
            del input_ids, res
            torch.cuda.empty_cache()
        experts = {i: logger.get_experts() for i, logger in loggers.items()}
        for logger in loggers.values():
            logger.freeze()
    return experts

def write_activations(train_data, test_data, loggers):
    act_dir = f"{DATA_DIR}/activations"
    os.makedirs(act_dir, exist_ok=True)
    train_activations = get_experts(train_data, loggers)
    for i, activations in train_activations.items():
        torch.save(activations, f"{act_dir}/train_matrix_{i}.pt")
    test_activations = get_experts(test_data, loggers)
    for i, activations in test_activations.items():
        torch.save(activations, f"{act_dir}/test_matrix_{i}.pt")

write_activations(train_data, test_data, loggers)

# # Extra

# from matplotlib import pyplot as plt
# acts = torch.load(f"{DATA_DIR}/activations/train_matrix_0.pt")
# for i in range(acts.shape[0]):
#     acts[i, i] = 0
# acts /= acts.max()
# plt.imshow(acts)
# plt.show()

# corr = torch.corrcoef(model.model.layers[0].mlp.gate.gate.weight).detach().cpu()

# with torch.no_grad():
#     corr = torch.zeros_like(acts)
#     for _ in tqdm(range(10000)):
#         y = model.model.layers[0].mlp.gate.gate(torch.randn(100, model.model.layers[0].mlp.gate.gate.in_features, device=device))
#         _, selected = torch.topk(y, TOP_K)
#         selected = selected.cpu()
#         for s in selected:
#             for i in s:
#                 for j in s:
#                     corr[i, j] += 1

#     corr = corr.cpu().detach()

# for i in range(corr.shape[0]):
#     corr[i, i] = 0
# corr /= corr.max()
# plt.imshow(corr)
# plt.show()
# plt.imshow((acts - corr) ** 2)
# plt.show()
# print(torch.sum((acts - corr) ** 2))

# def build_matches(corr_matrix):
#     """Get a matrix of correlations and build
#     matching with most sum of correlations (greedy)"""
#     matches = []
#     n = corr_matrix.shape[0]
#     used = torch.zeros(n, dtype=torch.bool)

#     sim_matrix = corr_matrix.clone()

#     sim_matrix.fill_diagonal_(-float("inf"))

#     for _ in range(n // 2):
#         _, max_idx = torch.max(sim_matrix.view(-1), 0)
#         i = max_idx.item() // n
#         j = max_idx.item() % n

#         matches.append((i, j))
#         used[i] = True
#         used[j] = True

#         sim_matrix[i, :] = -float("inf")
#         sim_matrix[:, i] = -float("inf")
#         sim_matrix[j, :] = -float("inf")
#         sim_matrix[:, j] = -float("inf")

#     del sim_matrix, used
#     return matches

# def eval_match(corr_matrix, matches, top_k=8):
#     win = 0
#     for expert1, expert2 in matches:
#         win += corr_matrix[expert1, expert2]
#     return (win / (corr_matrix.sum() / top_k)).item()

# matches = build_matches(acts)
# print("Test matches with greed algorithm:", eval_match(acts, matches))

# matches = build_matches(corr)
# print("Test matches with greed algorithm:", eval_match(acts, matches))

# def main(train_data, test_data, draw=True):
#     with torch.no_grad():
#         res = []
#         best_win = 0
#         best_pair = []
#         for ind in tqdm(range(200)):
#             pairs = (
#                 torch.arange(0, train_data.shape[0])
#                 [torch.randperm(train_data.shape[0])]
#                 .reshape((-1, 2))
#                 .tolist()
#             )
#             pairs = [(x, y) for x, y in pairs]
#             win = eval_match(train_data, pairs)
#             friend = [-1] * train_data.shape[0]
#             for i, j in pairs:
#                 friend[i] = j
#                 friend[j] = i

#             n = train_data.shape[0]

#             def swap(i, j):
#                 if (i, friend[i]) in pairs:
#                     pairs.remove((i, friend[i]))
#                 else:
#                     pairs.remove((friend[i], i))
#                 if (j, friend[j]) in pairs:
#                     pairs.remove((j, friend[j]))
#                 else:
#                     pairs.remove((friend[j], j))
#                 pairs.append((i, j))
#                 pairs.append((friend[i], friend[j]))
#                 fi = friend[i]
#                 fj = friend[j]
#                 friend[fi] = fj
#                 friend[fj] = fi
#                 friend[i] = j
#                 friend[j] = i

#             def unique(data: list):
#                 data.extend([friend[i] for i in data])
#                 return len(set(data)) == len(data)

#             def win_from_swap(i, j):
#                 return (
#                     train_data[i, j] + train_data[friend[i], friend[j]]
#                     - train_data[i, friend[i]] - train_data[j, friend[j]]
#                 ).item()

#             while True:
#                 delta = 0
#                 i, j = -1, -1
#                 for i1 in range(n):
#                     for j1 in range(i1 + 1, n):
#                         new_delta = win_from_swap(i1, j1)
#                         if new_delta > 0:
#                             delta = new_delta
#                             i, j = i1, j1
#                 if delta < 1e-6:
#                     break

#                 swap(i, j)

#             win = eval_match(train_data, pairs)
#             res.append(win)
#             if win > best_win:
#                 best_win = win
#                 best_pair = pairs
#                 print(ind, win)
#         if draw:
#             plt.hist(res)
#             plt.show()
#             plt.plot(res)
#             plt.show()

#     print("Train", eval_match(train_data, best_pair))
#     print("Test", eval_match(test_data, best_pair))
#     return best_pair

# matches = main(acts, acts, False)
# print("Test matches with greed algorithm:", eval_match(acts, matches))

# matches = main(corr, acts, False)
# print("Test matches with greed algorithm:", eval_match(acts, matches))

# torch.cosine_similarity(corr.flatten(), acts.flatten(), dim=0)


## Этап 2. Разбиение экспертов на пары


In [ ]:
import os
from datetime import datetime

import torch
from matplotlib import pyplot as plt

print(f"PAIRING_METHOD = {PAIRING_METHOD}")

set_seed(SEED)

if PAIRING_METHOD == "activation_correlation":
    sim0 = compute_activation_correlation_matrix(LAYERS[0], DATA_DIR)
    now = datetime.now()
    _plot_dir = f"plots/activation_correlation/{now.strftime('%Y_%m_%d_%H_%M_%S')}"
    os.makedirs(_plot_dir, exist_ok=True)
    plt.imshow(sim0)
    plt.savefig(f"{_plot_dir}/train_layer{LAYERS[0]}.png")
    plt.close()

evolution_steps = EVOLUTION_STEPS

best_pairs = {}
for layer in LAYERS:
    pairs = get_pairs(
        method=PAIRING_METHOD,
        layer=layer,
        n_experts=EXPERTS,
        data_dir=DATA_DIR,
        device=device,
        evolution_steps=evolution_steps,
        progress=True,
    )
    best_pairs[layer] = pairs
    print(f"layer {layer}: {len(pairs)} pairs")

os.makedirs(f"{DATA_DIR}/pairs", exist_ok=True)
for layer in LAYERS:
    torch.save(best_pairs[layer], f"{DATA_DIR}/pairs/{layer}.pt")

print("Saved data example:", torch.load(f"{DATA_DIR}/pairs/{LAYERS[0]}.pt"))


## Этап 3. Сохранение экспертов


In [ ]:
import torch
import os
import transformers
from tqdm import tqdm

device = torch.device("cpu")

set_seed(SEED)

if "model" not in globals():
    model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map=device
    )

for n_layer in tqdm(LAYERS):
    experts = model.model.layers[n_layer].mlp.experts
    os.makedirs(f"{DATA_DIR}/experts/{n_layer}", exist_ok=True)

    try:
        for i, expert in enumerate(experts):
            torch.save(
                turn_in_mlp(expert, device=torch.device("cpu")).to(dtype=model.dtype),
                f"{DATA_DIR}/experts/{n_layer}/expert_{i}.pt",
            )
    except TypeError:
        for i in range(experts.num_experts):
            new_mlp = AdvancedMLP(
                experts.hidden_dim,
                experts.intermediate_dim,
                experts.hidden_dim,
                device=torch.device("cpu"),
            )
            new_mlp.gate.weight = torch.nn.Parameter(
                experts.gate_up_proj[i, : experts.intermediate_dim].detach().clone()
            )
            new_mlp.up.weight = torch.nn.Parameter(
                experts.gate_up_proj[i, experts.intermediate_dim :].detach().clone()
            )
            new_mlp.down.weight = torch.nn.Parameter(
                experts.down_proj[i].detach().clone()
            )
            torch.save(
                new_mlp.to(dtype=model.dtype),
                f"{DATA_DIR}/experts/{n_layer}/expert_{i}.pt",
            )

    torch.save(
        model.model.layers[n_layer].mlp.gate.cpu(),
        f"{DATA_DIR}/experts/{n_layer}/gate.pt",
    )
    torch.save(
        model.model.layers[n_layer].mlp.shared_expert.cpu(),
        f"{DATA_DIR}/experts/{n_layer}/shared_expert.pt",
    )
    torch.save(
        model.model.layers[n_layer].mlp.shared_expert_gate.cpu(),
        f"{DATA_DIR}/experts/{n_layer}/shared_expert_gate.pt",
    )


## Этап 4. Сохранение входов MLP


In [ ]:
import torch
import torch.nn as nn
import os
from tqdm import tqdm
import transformers
import datasets

set_seed(SEED)

if "model" not in globals():
    model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map=device
    )
if "tokenizer" not in globals():
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

# Remove extra layers
MAX_LAYERS = max(LAYERS) + 1
for i in range(MAX_LAYERS, len(model.model.layers)):
    del model.model.layers[MAX_LAYERS]

torch.cuda.empty_cache()
model = model.to(device)

if "_train_data" not in globals():
    _train_data = datasets.load_dataset(**TRAIN_DATASET_NAME)["question"]
if "_test_data" not in globals():
    _test_data = datasets.load_dataset(**TEST_DATASET_NAME)["question"]
train_data = _train_data[:TRAIN_SIZE]
test_data = _test_data[:TEST_SIZE]

class Logger(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.data = []

    def forward(self, *args, **kwargs):
        for x in args[0].detach().cpu()[0, :, :]:
            self.data.append(x)
        return self.model.forward(*args, **kwargs)

    def get(self):
        res = self.data
        self.data = []
        return res

for n_layer in LAYERS:
    logger = Logger(model.model.layers[n_layer].mlp)
    model.model.layers[n_layer].mlp = logger
    os.makedirs(f"{DATA_DIR}/mlp_inputs", exist_ok=True)

    for data in tqdm(train_data):
        input_ids = tokenizer(data, return_tensors="pt")["input_ids"]
        model.forward(input_ids.to(device))
    torch.save(logger.get(), f"{DATA_DIR}/mlp_inputs/train_{n_layer}.pt")

    for data in tqdm(test_data):
        input_ids = tokenizer(data, return_tensors="pt")["input_ids"]
        model.forward(input_ids.to(device))
    torch.save(logger.get(), f"{DATA_DIR}/mlp_inputs/test_{n_layer}.pt")

    model.model.layers[n_layer].mlp = logger.model


## Этап 5. Активации для каждой пары


In [ ]:
import torch
import os
from tqdm import tqdm
import transformers

set_seed(SEED)

if "model" not in globals():
    model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map=device
    )
if "tokenizer" not in globals():
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

# Remove extra layers
MAX_LAYERS = max(LAYERS) + 1
for i in range(MAX_LAYERS, len(model.model.layers)):
    del model.model.layers[MAX_LAYERS]

torch.cuda.empty_cache()
model = model.to(device)

train_data = {}
test_data = {}
for layer in LAYERS:
    train_data[layer] = torch.load(f"{DATA_DIR}/mlp_inputs/train_{layer}.pt")
    test_data[layer] = torch.load(f"{DATA_DIR}/mlp_inputs/test_{layer}.pt")

def count_activations(data, layer):
    res = []
    for example in tqdm(data):
        router_output = model.model.layers[layer].mlp.gate(example.to(device))
        if isinstance(router_output, tuple):
            selected = router_output[-1]
        else:
            _, selected = router_output.topk(TOP_K, dim=-1)

        res.append(selected.cpu().flatten())
    return res

os.makedirs(f"{DATA_DIR}/mlp_inputs", exist_ok=True)

for layer in LAYERS:
    data = count_activations(train_data[layer], layer)
    torch.save(data, f"{DATA_DIR}/activations/train_all_{layer}.pt")

    data = count_activations(test_data[layer], layer)
    torch.save(data, f"{DATA_DIR}/activations/test_all_{layer}.pt")

train_activations = {}
test_activations = {}
for layer in LAYERS:
    train_activations[layer] = torch.load(
        f"{DATA_DIR}/activations/train_all_{layer}.pt"
    )
    test_activations[layer] = torch.load(f"{DATA_DIR}/activations/test_all_{layer}.pt")

pairs = {}
pairs_ind = {}
for layer in LAYERS:
    pairs[layer] = torch.load(f"{DATA_DIR}/pairs/{layer}.pt")
    pairs_ind[layer] = [-1] * (2 * len(pairs[layer]))
    for i, (x, y) in enumerate(pairs[layer]):
        pairs_ind[layer][x] = i
        pairs_ind[layer][y] = i

def get_pairs_activations(activations, layer):
    pairs_activations = []
    for _ in range(len(pairs[layer])):
        pairs_activations.append(set())

    for ind, activation in tqdm(enumerate(activations), total=len(activations)):
        topk = len(activation)
        for i in range(topk):
            activated_expert = pairs_ind[layer][activation[i]]
            pairs_activations[activated_expert].add(ind)
        # for i in range(topk):
        #     for j in range(i + 1, topk):
        #         if (pairs_ind[layer][activation[i]]
        #                 == pairs_ind[layer][activation[j]]):
        #             activated_expert = pairs_ind[layer][activation[i]]
        #             pairs_activations[activated_expert].append(ind)

    return list(map(list, pairs_activations))

for layer in LAYERS:
    torch.save(
        get_pairs_activations(train_activations[layer], layer),
        f"{DATA_DIR}/activations/pairs_train_{layer}.pt",
    )
    torch.save(
        get_pairs_activations(test_activations[layer], layer),
        f"{DATA_DIR}/activations/pairs_test_{layer}.pt",
    )


## Этап 6. Сжатие экспертов (fusion)


In [ ]:
# # Imports

import torch
from datetime import datetime
import torch.nn as nn
import copy
from matplotlib import pyplot as plt
from tqdm import tqdm
import time
import os
import seaborn as sns
import numpy as np

if FILTER_FUSIONS is not None:
    FUSION_METHODS = {k: v for k, v in FUSION_METHODS.items() if k in FILTER_FUSIONS}
if FILTER_METRICS is not None:
    best_metrics = {k: v for k, v in best_metrics.items() if k in FILTER_METRICS}

set_seed(SEED)

# # Classes

class Logger(nn.Module):
    """Simple class for saving passing values"""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model
        self.xs: list[torch.Tensor] | None = None

    def forward(self, x: torch.Tensor):
        if self.xs is not None:
            self.xs.append(x)
        return self.model(x)

# # Fusion

@torch.no_grad()
def compare_fusions_error_and_time(
    mlp1, mlp2, train_data, test_data, fusion_experts=None, ind=None
) -> dict[str, dict[str, list[float]]]:
    """Take two experts from model first layer
    and count MSE and time for existed fusion methods"""
    fusions = {}
    for fusion_name, fusion_methos in FUSION_METHODS.items():
        fusions[fusion_name] = fusion_methos(mlp1, mlp2, train_data)
    if fusion_experts is not None:
        fusion_experts[ind] = fusions

    for metric, inverse in best_metrics.items():
        fusions[f"Best_{metric}"] = fusions["Projection"]
    if len(test_data) > 10:
        results = {
            fusion_name: test_model_on_data(fusions[fusion_name], mlp1, mlp2, test_data)
            for fusion_name, _ in FUSION_METHODS.items()
        }
        for metric, inverse in best_metrics.items():
            best_name = list(results.keys())[0]
            for fusion_name in FUSION_METHODS.keys():
                if (
                    results[fusion_name][metric] > results[best_name][metric]
                ) == inverse:
                    best_name = fusion_name
            fusions[f"Best_{metric}"] = fusions[best_name]
            results[f"Best_{metric}"] = results[best_name]

        return results
    else:
        return None

experiment_results: list[dict[str, dict[str, list[float]]]] = []
act_dir = f"{DATA_DIR}/activations"
for layer in tqdm(LAYERS):
    experts = [
        torch.load(f"{DATA_DIR}/experts/{layer}/expert_{i}.pt", weights_only=False)
        for i in range(EXPERTS)
    ]
    train_data = torch.load(f"{DATA_DIR}/mlp_inputs/train_{layer}.pt")
    test_data = torch.load(f"{DATA_DIR}/mlp_inputs/test_{layer}.pt")
    pairs = torch.load(f"{DATA_DIR}/pairs/{layer}.pt")
    train_pairs_activations = torch.load(
        f"{DATA_DIR}/activations/pairs_train_{layer}.pt"
    )
    test_pairs_activations = torch.load(f"{DATA_DIR}/activations/pairs_test_{layer}.pt")

    fusion_experts = [None] * len(pairs)

    for i in tqdm(range(len(pairs))):
        expert_1, expert_2 = pairs[i]
        train_data_for_pair = [
            train_data[ind].to(device) for ind in train_pairs_activations[i]
        ]
        test_data_for_pair = [
            test_data[ind].to(device) for ind in test_pairs_activations[i]
        ]
        res = compare_fusions_error_and_time(
            experts[expert_1].to(device),
            experts[expert_2].to(device),
            train_data_for_pair,
            test_data_for_pair,
            fusion_experts,
            i,
        )
        if len(test_data_for_pair) > 10:
            experiment_results.append(res)

    for x in train_data:
        del x
    for x in test_data:
        del x
    for i, fused in enumerate(fusion_experts):
        for fusion_name, fusion_expert in fused.items():
            os.makedirs(
                f"{DATA_DIR}/fusion_experts/{fusion_name}/{layer}", exist_ok=True
            )
            while True:
                try:
                    torch.save(
                        fusion_expert.cpu(),
                        f"{DATA_DIR}/fusion_experts/{fusion_name}/{layer}/{i}.pt",
                    )
                    break
                except Exception as e:
                    print(f"[DEBUG] {e}")
            del fusion_expert
    torch.cuda.empty_cache()

now = datetime.now()
time = now.strftime("%Y_%m_%d_%H_%M_%S")
_plot_dir = f"plots/fusion/{time}"
os.makedirs(_plot_dir, exist_ok=True)

os.makedirs(f"logs/fusion", exist_ok=True)
torch.save(experiment_results, f"logs/fusion/{time}.pt")

experiment_results = torch.load(f"logs/fusion/{time}.pt")

metrics = list(best_metrics.keys()) + ["Time"]
log_safe_metrics = {
    "MSE",
    "RMSE",
    "MAE",
    "RelMSE",
    "JS",
    "KL",
    "CKA",
    "EntropyGap",
    "Time",
}
data = {
    metric: {
        fusion_name: [
            experiment_result[fusion_name][metric]
            for experiment_result in experiment_results
        ]
        for fusion_name in FUSION_METHODS.keys()
    }
    for metric in metrics
}

experts = len(data[metrics[0]][list(FUSION_METHODS.keys())[0]]) // len(LAYERS)

for metric in metrics:
    print(f"Meadian {metric} results for fusion methods:")
    for method_name in FUSION_METHODS.keys():
        print(f"For {method_name} - {np.median(data[metric][method_name]):.3f}")

for i, layer in enumerate(LAYERS):
    print()
    print(f"On layer {layer}:")
    for metric in metrics:
        print(f"{metric} results for fusion methods:")
        for method_name in FUSION_METHODS.keys():
            print(
                f"For {method_name} - {np.median(data[metric][method_name][i * experts : (i + 1) * experts]):.3f}"
            )

for values in data.values():
    for key in values.keys():
        values[key] = [value for value in values[key] if value < 1e6]

plt.figure(figsize=(6 * len(metrics), 6))
for i, metric in enumerate(metrics):
    plt.subplot(1, len(metrics), i + 1)
    plt.title(metric)
    for name, values in data[metric].items():
        plot_values = values
        if metric in log_safe_metrics and metric != "Cosine":
            plot_values = np.log(
                np.maximum(np.asarray(values, dtype=np.float64), 1e-12)
            )
        plt.hist(
            plot_values,
            label=name,
            bins=100,
            density=True,
            alpha=0.5,
            cumulative=True,
            histtype="stepfilled",
        )
    plt.legend()
    plt.title(metric)
plt.savefig(f"{_plot_dir}/hist_stepfilled_1.png")
plt.close()

plt.figure(figsize=(6 * len(metrics), 6))
for i, metric in enumerate(metrics):
    plt.subplot(1, len(metrics), i + 1)
    plt.title(metric)
    for name, values in data[metric].items():
        plot_values = values
        if metric in log_safe_metrics and metric != "Cosine":
            plot_values = np.log(
                np.maximum(np.asarray(values, dtype=np.float64), 1e-12)
            )
        plt.hist(
            plot_values,
            label=name,
            bins=100,
            density=True,
            alpha=0.5,
            cumulative=True,
            histtype="stepfilled",
        )
    plt.legend()
    plt.title(metric)
plt.savefig(f"{_plot_dir}/hist_stepfilled_2.png")
plt.close()

plt.figure(figsize=(6 * len(metrics), 6))
for i, metric in enumerate(metrics):
    plt.subplot(1, len(metrics), i + 1)
    plt.title(metric)
    names = list(data[metric].keys())

    handles = []
    for j, name in enumerate(names):
        values = data[metric][name]
        if metric in log_safe_metrics and metric != "Cosine":
            values = np.log(np.maximum(np.asarray(values, dtype=np.float64), 1e-12))
        vp = plt.violinplot(values, positions=[j])
        handles.append(vp["bodies"][0])

    plt.legend(handles, names)
    plt.title(metric)

plt.savefig(f"{_plot_dir}/violinplot.png")
plt.close()

plt.figure(figsize=(6 * len(metrics), 6))
for i, metric in enumerate(metrics):
    plt.subplot(1, len(metrics), i + 1)
    plt.title(metric)
    names = list(data[metric].keys())

    handles = []
    for j, name in enumerate(names):
        values = data[metric][name]
        if metric in log_safe_metrics and metric != "Cosine":
            values = np.log(np.maximum(np.asarray(values, dtype=np.float64), 1e-12))
        vp = plt.violinplot(values, positions=[j])
        handles.append(vp["bodies"][0])

    plt.legend(handles, names)
    plt.title(metric)

plt.savefig(f"{_plot_dir}/violinplot_log.png")
plt.close()

plt.figure(figsize=(6 * len(metrics), 6))
for i, metric in enumerate(metrics):
    plt.subplot(1, len(metrics), i + 1)
    plt.title(metric)
    names = list(data[metric].keys())
    if metric in log_safe_metrics and metric != "Cosine":
        sns.kdeplot(
            [
                np.log(
                    np.maximum(np.asarray(data[metric][key], dtype=np.float64), 1e-12)
                )
                for key in names
            ],
            fill=True,
            alpha=0.3,
        )
    else:
        sns.kdeplot([data[metric][key] for key in names], fill=True, alpha=0.3)
    plt.legend(names)
plt.savefig(f"{_plot_dir}/kdeplot.png")
plt.close()


## Этап 7. Бенчмарк fused vs original


In [ ]:
import torch
from matplotlib import pyplot as plt
from tqdm import tqdm
import time
import seaborn as sns
from datetime import datetime
import os

set_seed(SEED)

LAYER = LAYERS[0]

train_data = torch.load(f"{DATA_DIR}/mlp_inputs/train_{LAYER}.pt")
test_data = torch.load(f"{DATA_DIR}/mlp_inputs/test_{LAYER}.pt")

model = get_fusion_layer_by_index(LAYER, f"fusion_experts/ZipIt", device=device)

batch_size = 128
total = 100

with torch.no_grad():
    with torch.autocast("cuda"):
        for ind in tqdm(range(0, batch_size * total, batch_size)):
            batch = torch.stack(train_data[ind : ind + batch_size])
            batch = batch.to(device)
            model.forward(batch)[0]

with torch.no_grad():
    with torch.autocast("cuda"):
        for ind in tqdm(range(0, batch_size * total, batch_size)):
            batch = torch.stack(train_data[ind : ind + batch_size])
            batch = batch.to(device)
            model.forward_default(batch)[0]

norm = []
err = []
cos_sim = []

time_new = []
time_base = []

with torch.no_grad():
    with torch.autocast("cuda"):
        for ind in tqdm(range(0, batch_size * total, batch_size)):
            input = torch.stack(train_data[ind : ind + batch_size]).to(device)
            time_start = time.perf_counter()
            x = model.forward(input)[0][0]
            time_middle = time.perf_counter()
            y = model.forward_default(input)[0][0]
            time_end = time.perf_counter()
            for i in range(y.shape[0]):
                norm.append(y[i].square().sum().sqrt().cpu().detach().item())
                err.append((x[i] - y[i]).square().sum().sqrt().cpu().detach().item())
                cos_sim.append(
                    torch.cosine_similarity(x[i].flatten(), y[i].flatten(), dim=0)
                    .cpu()
                    .detach()
                    .item()
                )
            time_new.append(time_middle - time_start)
            time_base.append(time_end - time_middle)

print("Norm:", norm[0])
print("Error:", err[0])
print("Cos_sim:", cos_sim[0])

now = datetime.now()
time = now.strftime("%Y_%m_%d_%H_%M_%S")
_plot_dir = f"plots/fusion/{time}"
os.makedirs(_plot_dir, exist_ok=True)

sns.kdeplot(cos_sim, label="Cosine similarity", fill=True, alpha=0.3, log_scale=True)
# plt.xlabel('Norm')
# plt.ylabel('Density')
# plt.title('Density of norms')
# plt.legend()
# plt.grid(True, alpha=0.3)

plt.savefig(f"{_plot_dir}/kdeplot.png")
plt.close()

sns.kdeplot(
    [1e-10 + x for x in norm],
    label="Norm of the true vector",
    fill=True,
    alpha=0.3,
    log_scale=True,
)
sns.kdeplot(
    [1e-10 + x for x in err if x != 0],
    label="Norm of the error vector",
    fill=True,
    alpha=0.3,
    log_scale=True,
)
plt.xlabel("Norm")
plt.ylabel("Density")
plt.title("Density of norms")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(f"{_plot_dir}/kdeplot_norms.png")
plt.close()

sns.kdeplot(time_new, label="Our method", fill=True, alpha=0.3)
sns.kdeplot(time_base, label="Original model", fill=True, alpha=0.3)
plt.xlabel("Time")
plt.ylabel("Density")
plt.title("Density of time")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(f"{_plot_dir}/kdeplot_times.png")
plt.close()


## Этап 8. Примеры генерации


In [ ]:
import torch
from tqdm import tqdm
import transformers
import datasets

EXAMPLES = 10

set_seed(SEED)

if "model" not in globals():
    model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME)
if "tokenizer" not in globals():
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = model.to(device)

if "_test_data" not in globals():
    _test_data = datasets.load_dataset(**TEST_DATASET_NAME)["question"]
test_data = _test_data[:EXAMPLES]

@torch.autocast("cuda")
def generate_examples(model, examples):
    res = []
    for example in tqdm(examples):
        outputs = model.generate(**example, max_new_tokens=10)
        res.append((tokenizer.decode(outputs[0])))
    return res

examples = []
for ind in range(EXAMPLES):
    messages = [
        {"role": "user", "content": test_data[ind]},
    ]
    examples.append(
        tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
    )

base_examples = generate_examples(model, examples)

for layer in LAYERS:
    del model.model.layers[layer].mlp
    torch.cuda.empty_cache()
    new_mlp = get_fusion_layer_by_index(
        layer,
        "fusion_experts/ZipIt",
        device=device,
        use_default_experts=False,
    ).to(device)
    model.model.layers[layer].mlp = new_mlp

new_examples = generate_examples(model, examples)

for base_generation, new_generation in zip(base_examples, new_examples):
    print(
        base_generation,
        "\n" + "+" * 100 + "\n",
        new_generation,
        "\n" + "-" * 100 + "\n",
    )


## Этап 9. Оценка точности на QA


In [ ]:
import os
import torch
import transformers
from matplotlib import pyplot as plt
from tqdm import tqdm
import gc

model_id = "Qwen/Qwen1.5-MoE-A2.7B-Chat"
use_default_experts = False

def _dataset_tag(ds):
    return f"{ds['path'].split('/')[-1]}_{ds['split']}"

TEST_RESULTS_DIR = (
    f"test_result_{_dataset_tag(TRAIN_DATASET_NAME)}_{_dataset_tag(TEST_DATASET_NAME)}"
)

if FILTER_FUSIONS is not None:
    FUSION_METHODS = {k: v for k, v in FUSION_METHODS.items() if k in FILTER_FUSIONS}
if FILTER_METRICS is not None:
    best_metrics = {k: v for k, v in best_metrics.items() if k in FILTER_METRICS}

set_seed(SEED)

def free_model(model):
    try:
        model.cpu()
    except Exception:
        pass
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

base_accuracy = None
base_size = None

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class ZeroLayer(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, *args, **kwargs):
        return torch.zeros_like(args[0])

def zero_fusion(model, layer):
    return ZeroLayer().to(model.device, dtype=model.dtype)

def default_fusion(name):
    def inner(model, layer):
        new_layer = get_fusion_layer_by_index(
            layer,
            f"fusion_experts/{name}",
            device=model.device,
            use_default_experts=use_default_experts,
        ).to(model.device, dtype=model.dtype)
        for expert in new_layer.fusion_experts:
            if expert.U is not None:
                expert.U = expert.U.to(model.device, dtype=model.dtype)
        return new_layer

    return inner

experts_fusions = {name: default_fusion(name) for name in FUSION_METHODS.keys()}
experts_fusions["zero"] = zero_fusion
for best_metric in best_metrics.keys():
    experts_fusions[f"Best_{best_metric}"] = default_fusion(f"Best_{best_metric}")

for fusion_name, experts_fusion in experts_fusions.items():
    print(f"Testing {fusion_name}")
    model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map=device
    )

    if base_accuracy is None:
        base_accuracy = evaluate_qa(
            model, tokenizer, TEST_DATASET_NAME, VALIDATION_SIZE, seed=SEED
        )
        base_size = count_params(model)
    accuracy_list = [base_accuracy]
    size_list = [base_size]

    for layer in tqdm(LAYERS):
        if REPLACE_LAYERS_SEQUANTIAL:
            del model.model.layers[layer].mlp
            torch.cuda.empty_cache()
            model.model.layers[layer].mlp = experts_fusion(model, layer)
            accuracy_list.append(
                evaluate_qa(
                    model, tokenizer, TEST_DATASET_NAME, VALIDATION_SIZE, seed=SEED
                )
            )
            size_list.append(count_params(model))
        else:
            old_mlp = model.model.layers[layer].mlp
            model.model.layers[layer].mlp = experts_fusion(model, layer)
            accuracy_list.append(
                evaluate_qa(
                    model, tokenizer, TEST_DATASET_NAME, VALIDATION_SIZE, seed=SEED
                )
            )
            size_list.append(count_params(model))
            del model.model.layers[layer].mlp
            torch.cuda.empty_cache()
            model.model.layers[layer].mlp = old_mlp

    mode_str = "seq" if REPLACE_LAYERS_SEQUANTIAL else "single"
    print(f"model's accuracy list after {fusion_name} ({mode_str}):", accuracy_list)
    os.makedirs(f"{TEST_RESULTS_DIR}/accuracy", exist_ok=True)
    os.makedirs(f"{TEST_RESULTS_DIR}/model_size", exist_ok=True)
    torch.save(
        accuracy_list,
        f"{TEST_RESULTS_DIR}/accuracy/accuracy_list_{mode_str}_{fusion_name}.pt",
    )
    torch.save(
        size_list,
        f"{TEST_RESULTS_DIR}/model_size/model_size_list_{mode_str}_{fusion_name}.pt",
    )

    if REPLACE_BEST_LAYERS_SEQUANTIAL:
        best_layers = [
            x[0] for x in sorted(enumerate(accuracy_list[1:]), key=lambda x: -x[1])
        ]
        print("Best layers:", best_layers)
        new_accuracy_list = [base_accuracy]
        new_size_list = [base_size]

        for layer in tqdm(best_layers):
            del model.model.layers[layer].mlp
            torch.cuda.empty_cache()
            model.model.layers[layer].mlp = experts_fusion(model, layer)
            new_accuracy_list.append(
                evaluate_qa(
                    model, tokenizer, TEST_DATASET_NAME, VALIDATION_SIZE, seed=SEED
                )
            )
            new_size_list.append(count_params(model))
        print(f"model's accuracy list after {fusion_name} (seq):", new_accuracy_list)
        torch.save(
            new_accuracy_list,
            f"{TEST_RESULTS_DIR}/accuracy/accuracy_list_seq_{fusion_name}.pt",
        )
        torch.save(
            new_size_list,
            f"{TEST_RESULTS_DIR}/model_size/model_size_list_seq_{fusion_name}.pt",
        )
    free_model(model)
    model = None

if REPLACE_LAYERS_SMART_SEQUANTIAL:
    for fusion_name, experts_fusion in tqdm(list(experts_fusions.items())):
        changed = set()
        remain = set(LAYERS)
        print(f"Testing {fusion_name}")
        model = transformers.AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, device_map=device
        )
        if base_accuracy is None:
            base_accuracy = evaluate_qa(
                model, tokenizer, TEST_DATASET_NAME, VALIDATION_SIZE, seed=SEED
            )
            base_size = count_params(model)
        accuracy_list = [base_accuracy]
        size_list = [base_size]
        for _ in tqdm(LAYERS):
            best_ind = -1
            best_score = -1

            for layer in remain:
                old_mlp = model.model.layers[layer].mlp
                model.model.layers[layer].mlp = experts_fusion(model, layer)
                new_score = evaluate_qa(
                    model, tokenizer, TEST_DATASET_NAME, VALIDATION_SIZE, seed=SEED
                )
                if new_score > best_score:
                    best_score = new_score
                    best_ind = layer
                del model.model.layers[layer].mlp
                torch.cuda.empty_cache()
                model.model.layers[layer].mlp = old_mlp
            remain.remove(best_ind)
            changed.add(best_ind)
            model.model.layers[best_ind].mlp = experts_fusion(model, best_ind)
            accuracy_list.append(best_score)
            size_list.append(count_params(model))
        print(f"model's accuracy list after {fusion_name} smart seq:", accuracy_list)
        os.makedirs(f"{TEST_RESULTS_DIR}/accuracy", exist_ok=True)
        os.makedirs(f"{TEST_RESULTS_DIR}/model_size", exist_ok=True)
        torch.save(
            accuracy_list,
            f"{TEST_RESULTS_DIR}/accuracy/accuracy_list_smart_seq_{fusion_name}.pt",
        )
        torch.save(
            size_list,
            f"{TEST_RESULTS_DIR}/model_size/model_size_list_smart_seq_{fusion_name}.pt",
        )
        free_model(model)
        model = None


## Быстрый прогон (опционально)

Для отладки на одном GPU уменьшите масштаб **в первой ячейке (конфиг)**:
```python
LAYERS = [0]
TRAIN_SIZE = 64
TEST_SIZE = 64
VALIDATION_SIZE = 100
EVOLUTION_STEPS = 5
FILTER_FUSIONS = ["ZipIt", "Average"]
```
Полный прогон на 24 слоях занимает много времени и VRAM.
